# SGR Agent Core - Мастер-класс (AI Conf)

> **Цель занятия:** провести аудиторию по пути от бизнес-запроса ("хочу агента по локальным файлам") до работающего OpenAI-совместимого сервиса за 115-120 минут.
> **Фокус:** конфигурация, метрики, воспроизводимость - а не "магические промпты".

---

## Блок 0. Вступление (~5 мин)

### Что понадобится
- Python 3.11+, Docker (опционально)
- Git
- API-ключ OpenAI или доступ к OpenAI-совместимому эндпоинту
- (Опционально) Tavily API-ключ для Deep Research
- (Опционально) Langfuse аккаунт для observability

### Материалы и ссылки

| Ресурс | Ссылка |
|--------|--------|
| Репозиторий | `https://github.com/vamplabai/sgr-agent-core` |
| Docker-образ | `ghcr.io/vamplabai/sgr-agent-core:latest` |
| PyPI-пакет | `pip install sgr-agent-core` |
| Obsidian Agent Client | `https://github.com/RAIT-09/obsidian-agent-client` |
| Документация Agent Client | `https://rait-09.github.io/obsidian-agent-client/agent-setup/custom-agents.html` |

### Обещанный результат
К концу занятия у вас будет локальный файловый агент с HTTP API, который можно подключить к Obsidian.

**Формат:** теория -> live demo -> практика участников

In [ ]:
# Проверим структуру репозитория
!tree -L 2 .

---

## Блок 1. Что такое SGR Agent Core (~10 мин)

### Learning Objectives
- Объяснить, что такое Schema Guided Reasoning
- Показать разницу между SGR-агентом и обычным чат-ботом с tool calling
- Показать стабильность и валидируемость результатов

### Schema Guided Reasoning

SGR - это **явное описание шагов рассуждения**. Каждый шаг - это структурированный выбор, который можно валидировать.

**Чем SGR отличается от обычного агента:**
- Классический агент: непрозрачная цепочка вызовов функций
- SGR-агент: явные фазы (рассуждение -> планирование -> действия -> ответ)

### Прямое использование из Python

In [ ]:
import asyncio
from openai import AsyncOpenAI
from sgr_agent_core import AgentConfig
from sgr_agent_core.agents.sgr_agent import SGRAgent
from sgr_agent_core.tools import FinalAnswerTool


async def main() -> None:
    client = AsyncOpenAI(api_key="YOUR_OPENAI_API_KEY")

    agent = SGRAgent(
        task_messages=[
            {"role": "user", "content": "Собери краткий обзор по теме RAG систем"},
        ],
        openai_client=client,
        agent_config=AgentConfig(),
        toolkit=[FinalAnswerTool],
    )

    result = await agent.execute()
    print(result)


# Запускаем агента
# asyncio.run(main())  # Раскомментируйте для запуска

> **Ключевая идея:** SGR делает рассуждения агента детерминированными и интерпретируемыми. Это не просто еще один фреймворк - это попытка сделать AI-рассуждения прозрачными.

---

## Блок 2. Архитектура API сервера (~10 мин)

### Learning Objectives
- Понять архитектуру FastAPI-сервера
- Различать stateless и stateful режимы
- Понять жизненный цикл агента

### Архитектура

```
Клиент -> FastAPI Server -> Agent Storage -> Tools -> LLM -> Response
```

**Жизненный цикл агента:**
1. Создание (create)
2. Выполнение (execute)
3. Завершение (cleanup по таймауту)

### Stateless vs Stateful

| Режим | Описание | Когда использовать |
|-------|----------|-------------------|
| **Stateless** | Клиент отправляет полный контекст каждый раз | Простые запросы, нет истории |
| **Stateful** | Диалог по `agent_id`, история на сервере | Длинные диалоги, нужна память |

**Важно:** Stateful-агент живет в памяти сервера и имеет таймаут!

### Команды для тестирования API

In [ ]:
# Базовый стриминговый запрос (stateless)
!curl -N -X POST "http://localhost:8010/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "sgr_agent",
    "messages": [
      {"role": "user", "content": "Исследуй рынок RAG систем и сделай краткий вывод"}
    ],
    "stream": true
  }'

In [ ]:
# Stateful-запрос по agent_id
# После первого запроса используйте agent_id из ответа
!curl -N -X POST "http://localhost:8010/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "sgr_agent_12345678-1234-1234-1234-123456789012",
    "messages": [
      {"role": "user", "content": "Вот уточнение к предыдущему вопросу"}
    ],
    "stream": true
  }'

---

## Блок 3. Конфигурация YAML и типы агентов (~10 мин)

### Learning Objectives
- Понять иерархию конфигурации
- Уметь создавать минимальный `config.yaml`
- Различать типы агентов

### Иерархия конфигурации

```
defaults (в коде)
    ↓
env variables
    ↓
config.yaml (глобальный)
    ↓
agents.yaml (специфичный для агента)
```

### Типы агентов

| Тип | Описание | Когда использовать |
|-----|----------|-------------------|
| `SGRAgent` | Базовый, явные шаги рассуждения | Нужна прозрачность и контроль |
| `ToolCallingAgent` | Классический tool calling | Простые сценарии, скорость |
| `SGRToolCallingAgent` | Гибрид: SGR + tool calling | Баланс гибкости и структуры |

### Пример config.yaml

In [ ]:
config_yaml = '''
llm:
  api_key: "your-openai-api-key-here"
  base_url: "https://api.openai.com/v1"
  model: "gpt-4o-mini"
  max_tokens: 8000
  temperature: 0.4

execution:
  max_clarifications: 3
  max_iterations: 10
  mcp_context_limit: 15000
  logs_dir: "logs"

tools:
  web_search_tool:
    tavily_api_key: "your-tavily-api-key-here"
    tavily_api_base_url: "https://api.tavily.com"
    max_searches: 4
    max_results: 10
  extract_page_content_tool:
    tavily_api_key: "your-tavily-api-key-here"
    tavily_api_base_url: "https://api.tavily.com"
    content_limit: 1500
  final_answer_tool:
  clarification_tool:
  reasoning_tool:

agents:
  sgr_tool_calling_agent_no_reporting:
    base_class: "agents.ResearchSGRToolCallingAgentNoReporting"
    llm:
      model: "gpt-4o-mini"
      temperature: 0.4
    tools:
      - "web_search_tool"
      - "extract_page_content_tool"
      - "final_answer_tool"
      - "clarification_tool"
      - "reasoning_tool"
'''

# Сохраним конфиг
with open('config.yaml', 'w') as f:
    f.write(config_yaml)

print("Config saved to config.yaml")

> **Ключевая идея:** минимальные переопределения. Базовый конфиг задает LLM и execution, а конкретный агент только выбирает тулы и класс.

---

## Блок 4. Подготовка окружения (~5-10 мин)

### Learning Objectives
- Уметь запускать SGR двумя способами: Docker и pip
- Понять разницу между сервером и библиотекой

### Вариант 1: Docker (рекомендуется для быстрого старта)

In [ ]:
# Клонирование репозитория
!git clone https://github.com/vamplabai/sgr-agent-core.git
%cd sgr-agent-core

# Создание директорий для логов и отчетов
!sudo mkdir -p logs reports
!sudo chmod 777 logs reports

# Копирование примера конфига
!cp examples/sgr_deep_research/config.yaml.example examples/sgr_deep_research/config.yaml

In [ ]:
# Запуск через Docker (выполните в терминале)
docker_cmd = '''
docker run --rm -i \
  --name sgr-agent \
  -p 8010:8010 \
  -v "$(pwd)/examples/sgr_deep_research:/app/examples/sgr_deep_research:ro" \
  -v "$(pwd)/logs:/app/logs" \
  -v "$(pwd)/reports:/app/reports" \
  ghcr.io/vamplabai/sgr-agent-core:latest \
  --config-file /app/examples/sgr_deep_research/config.yaml \
  --host 0.0.0.0 \
  --port 8010
'''
print(docker_cmd)

### Вариант 2: pip install (больше контроля)

In [ ]:
# Создание виртуального окружения и установка
!python -m venv .venv
# Для Windows: !.venv\Scripts\activate
# Для Linux/Mac: source .venv/bin/activate

!pip install --upgrade pip
!pip install sgr-agent-core

# Копирование примера конфига
!cp config.yaml.example config.yaml

# Запуск сервера
# !sgr -c config.yaml  # Раскомментируйте для запуска

> **Выбор:** Docker - запуск за 2 минуты. pip - больше контроля, можно копаться в коде.

---

## Блок 5. Практика: Deep Research (~15 мин)

### Learning Objectives
- Запустить готового агента deep research
- Понять пайплайн: план -> поиск -> извлечение -> ответ
- Найти логи и отчеты

### Набор тулов для Deep Research

| Тул | Назначение |
|-----|------------|
| `GeneratePlan` | Создание плана исследования |
| `AdaptPlan` | Адаптация плана по ходу работы |
| `WebSearch` | Поиск в интернете |
| `ExtractPageContent` | Извлечение содержимого страниц |
| `FinalAnswer` | Формирование итогового ответа |

### Запуск

In [ ]:
# Подготовка конфига
!cd sgr-agent-core
!cp examples/sgr_deep_research/config.yaml.example examples/sgr_deep_research/config.yaml
# Отредактируйте config.yaml, добавив свои API ключи

# Запуск сервера
# !sgr -c examples/sgr_deep_research/config.yaml  # Раскомментируйте для запуска

In [ ]:
# Запрос к агенту
!curl -N -X POST "http://localhost:8010/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "sgr_tool_calling_agent",
    "messages": [
      {"role": "user", "content": "Сделай глубокое исследование рынка RAG решений"}
    ],
    "stream": true
  }'

### Где смотреть результаты

```
sgr-agent-core/
├── logs/          # Логи выполнения агентов
└── reports/       # Сгенерированные отчеты
```

> **Важно:** Обратите внимание на логи - там виден каждый шаг. Это не черный ящик, а прозрачный пайплайн!

---

## Блок 6. Практика: Файловый агент (~30-40 мин)

### Learning Objectives
- Перейти от research-сценария к локальному файловому агенту
- Понять границы безопасности (read-only)
- Освоить 4 режима работы агента

### 6.1 Файловые тулы из примера `sgr_file_agent`

| Тул | Назначение |
|-----|------------|
| `GetSystemPathsTool` | Стандартные пути: home, documents, downloads, desktop |
| `ListDirectoryTool` | Листинг с опцией рекурсии |
| `ReadFileTool` | Чтение с диапазоном строк |
| `SearchInFilesTool` | Поиск текста в файлах (grep-подобный) |
| `FindFilesFastTool` | Поиск файлов по паттерну/размеру/дате |

**Важно:** Работа с файлами - read-only по умолчанию! `working_directory` - это корень, за который агент не выйдет.

In [ ]:
# Подготовка файлового агента
!cd sgr-agent-core
!cp examples/sgr_file_agent/config.yaml.example examples/sgr_file_agent/config.yaml

### 6.2 Четыре режима работы агента

| Режим | Команда / Код | Когда использовать |
|-------|---------------|-------------------|
| **1. HTTP сервер** | `sgr --config-file examples/sgr_file_agent/config.yaml` | Интеграция с любыми OpenAI-совместимыми клиентами |
| **2. Stateful** | `model: "sgr_file_agent_<agent_id>"` в curl | Продолжение диалога без пересылки истории |
| **3. Библиотека** | `from sgr_agent_core.agents.sgr_agent import SGRAgent` | Встраивание в свой Python-код |
| **4. ACP (stdio)** | `sgracp --config /abs/path/to/config.yaml` | Интеграция с Obsidian, IDE и другими MCP-клиентами |

### 6.3 HTTP API с gpt-oss 120b

In [ ]:
# Конфиг для gpt-oss 120b
gpt_oss_config = '''
llm:
  api_key: "YOUR_KEY"
  base_url: "https://your-openai-compatible-host/v1"
  model: "gpt-oss:120b"
  max_tokens: 4000
  temperature: 0.3
'''

print(gpt_oss_config)

In [ ]:
# Запуск сервера
# !sgr --config-file examples/sgr_file_agent/config.yaml  # Раскомментируйте для запуска

# Запрос к файловому агенту
!curl -N -X POST "http://localhost:8010/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "sgr_file_agent",
    "messages": [
      {"role": "user", "content": "Найди в заметках все упоминания SGR за последний месяц"}
    ],
    "stream": true
  }'

### 6.4 ACP и Obsidian Agent Client

**ACP** - это мост между SGR и Obsidian.

**Проверка sgracp:**

In [ ]:
# Проверка что sgracp доступен
!which sgracp

# Запуск ACP сервера
# !sgracp --config /absolute/path/to/sgr-agent-core/examples/sgr_file_agent/config.yaml  # Раскомментируйте

In [ ]:
# Минимальный блок acp в yaml
acp_config = '''
acp:
  agent: sgr_file_agent
'''
print(acp_config)

**Установка плагина через BRAT:**
- BRAT: `https://github.com/RAIT-09/obsidian-agent-client`
- Документация: `https://rait-09.github.io/obsidian-agent-client/agent-setup/custom-agents.html`

> **Схема потока:** Obsidian -> Agent Client -> sgracp -> vault

---

## Блок 7. Метрики, тесты, Langfuse (~20 мин)

### Learning Objectives
- Понять уровни тестирования агента
- Уметь писать unit-тест на отдельный тул
- Понять, как подключить Langfuse для трейсинга

### Уровни тестирования

```
Unit (отдельный тул) -> Интеграция (с фикстурой каталога) -> E2E (полный агент)
```

### Пример теста тула

In [ ]:
test_code = '''
import pytest

from sgr_agent_core.agent_definition import AgentConfig
from sgr_agent_core.models import AgentContext

from examples.sgr_file_agent.tools.read_file_tool import ReadFileTool


@pytest.mark.asyncio
async def test_read_file_tool_smoke(tmp_path) -> None:
    f = tmp_path / "note.md"
    f.write_text("# hello", encoding="utf-8")
    tool = ReadFileTool(
        reasoning="read vault file",
        file_path=str(f),
    )
    context = AgentContext()
    config = AgentConfig()

    result = await tool(context, config)

    assert "hello" in result
'''

print(test_code)

In [ ]:
# Запуск тестов
!pytest examples/sgr_file_agent/tests/

### Метрики качества агента

| Метрика | Что измеряем |
|---------|-------------|
| Полнота ответа | Нашел ли агент все, что просили |
| Число лишних вызовов тулов | Эффективность рассуждений |
| Время выполнения | Latency, время на итерацию |
| Регрессия при смене модели | Сохраняется ли качество |

### Langfuse для трассировки

Langfuse позволяет:
- Трассировать вызовы тулов
- Измерять задержки
- Сравнивать прогоны
- Отслеживать стоимость

In [ ]:
# Пример конфига с Langfuse
langfuse_config = '''
observability:
  langfuse:
    enabled: true
    public_key: "your-public-key"
    secret_key: "your-secret-key"
    host: "https://cloud.langfuse.com"
'''

print(langfuse_config)

> **Совет:** Тестируйте не только "правильный" путь. Проверяйте, что агент graceful обрабатывает отсутствующие файлы, пустые директории, странные запросы.

---

## Блок 8. Домашнее задание (~5 мин)

### Варианты

1. **Файловый агент (рекомендуется):** закрепите папку с заметками, прогоните 3-5 запросов через API или ACP, сохраните конфиг
2. **Confluence-бот (альтернатива):** если в треке есть Confluence - повторите сценарий с Confluence API

### Что проверить
- [ ] Агент отвечает на запросы о содержимом папки
- [ ] Логи пишутся в `logs/`
- [ ] Конфиг сохранен и работает после перезапуска

> **Лучший способ закрепить** - запустить на своих данных. Не на демо-папке, а на своих заметках.

---

## Блок 9. Завершение и Roadmap (~5-10 мин)

### Learning Objectives
- Систематизировать пройденный материал
- Понять следующие шаги для развития файлового агента

### Резюме ключевых идей

```
конфигурация (YAML) -> сервер (HTTP) -> модель (LLM) -> ACP -> Obsidian
```

### Следующие шаги

- Тулы записи (write, append, mkdir)
- Отдельный конфиг под production
- MCP-интеграции
- Корпоративные политики доступа

### Идеи для самостоятельной работы

- Повторить цепочку на своем vault
- Измерить latency на разных моделях
- Написать 3 теста для своего агента

> **Итог:** Мы прошли путь от бизнес-запроса до работающего сервиса. Теперь у вас есть инструмент, который можно развивать: добавлять тулы, подключать к разным UI, измерять качество.

In [ ]:
# Финальная проверка изменений
!git diff

---

## Приложение: Шпаргалка по командам

```bash
# Клонирование
git clone https://github.com/vamplabai/sgr-agent-core.git
cd sgr-agent-core

# Deep Research
cp examples/sgr_deep_research/config.yaml.example examples/sgr_deep_research/config.yaml
sgr -c examples/sgr_deep_research/config.yaml

# Файловый агент
cp examples/sgr_file_agent/config.yaml.example examples/sgr_file_agent/config.yaml
sgr -c examples/sgr_file_agent/config.yaml

# ACP
sgracp --config /abs/path/to/config.yaml

# Тесты
pytest examples/sgr_file_agent/tests/
```

---

*Playbook актуален на 08.04.2026*